# Summary

Planning a pub crawl across central London presents a deceptively complex routing and resource-allocation problem. A group departing from any starting point must decide which pubs to visit, in what order, and how long to spend at each, subject to a finite time window, a limited budget, and the physical realities of walking. The naïve approach of picking nearby pubs at random ignores the interactions between route distance, spending, and enjoyment. This typically yields either an overly short crawl or one that overshoots its budget and time constraints.
This project frames the pub crawl as a route optimisation in which a traveller must select a subset of locations to visit along a route, maximising collected reward while respecting a travel-time budget. Our mixed integer program incorporates a multi-component objective that captures both enjoyment and fatigue, piecewise budget penalties that model diminishing marginal utility of spending, and a soft time-overrun penalty rather than a hard deadline. 

In [1]:
import subprocess, sys
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB,quicksum
import json
from OSMPythonTools.overpass import Overpass, overpassQueryBuilder
from OSMPythonTools.nominatim import Nominatim

## Querying pub locations

Pub location data was sourced from the OpenStreetMap (OSM) Overpass API. The raw JSON response yielded 3166 pubs, including columns such as latitude, longitude, and additional metadata such as food availability and type of cuisine served at the location. 
Starting point coordinates was obtained by converting the user inputted starting point post code into coordinates using postcode.io API. Then, the pairwise distance matrix was computed using the Haversine formula. The dataset was further reduced to a workable set from which the 100 nearest pubs to the starting point were selected. This produced a 101× 101 matrix of straight-line distances in kilometres, providing a consistent and computationally efficient proxy suitable for this scale of problem.


In [2]:
overpass = Overpass()
nominatim=Nominatim()

query = overpassQueryBuilder(
    area=nominatim.query('London'), 
    elementType=['node', 'way', 'relation'], 
    selector='"amenity"="pub"', 
    includeGeometry=True,
    out='body'
)

result=overpass.query(query)
print(f"Found {result.countElements()} pubs in London.")

data=result.toJSON()

df=pd.json_normalize(data['elements'])

pubs = {elem['id']: elem for elem in data['elements'] if elem['type'] == 'node'}
df.head()


Found 3166 pubs in London.


,type,id,lat,lon,tags.addr:city,tags.addr:housename,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.amenity,...,tags.note:addr:housenumber,tags.addr:unit:ref,tags.kids_area,tags.room,tags.check_date:currency:XBT,tags.currency:XBT,tags.seamark:hulk:category,tags.seamark:type,tags.ship:type,members
0,node,451152,51.600840,-0.194608,London,King of Prussia,363,N3 1DH,Regents Park Road,pub,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,451154,51.599579,-0.196028,NaN,NaN,319,NaN,Regents Park Road,pub,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,451271,51.614104,-0.176556,London,NaN,749,N12 0BP,High Road,pub,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,12242503,51.592016,0.027962,NaN,NaN,NaN,NaN,NaN,pub,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,15262028,51.516481,-0.169827,London,NaN,30,W2 1JQ,Southwick Street,pub,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Exploration

In [3]:
loc=df.copy()
loc.columns=[col.replace('tags.','') for col in loc]
loc = loc.loc[(loc['type'] == 'node').values, :]
loc=loc[['id','lat','lon','addr:postcode','food','name','cuisine','opening_hours']]


In [4]:
#Dropping NAs

loc_cleaned=loc[loc['name'].notna()]

loc_cleaned.shape
print(f'There are {loc_cleaned.shape} pubs in the dataset')

loc_cleaned.head()

#Creating lists for ids and names
pub_id= loc_cleaned['id'].astype(int).tolist()
pub_names=loc_cleaned['name'].tolist()

#Dictionary for longitude and latitudes
coordinates = (
    loc_cleaned[loc_cleaned['id'].isin(pub_id)]
    .set_index('id')[['lat', 'lon']]
    .T.to_dict('list')
)


There are (1261, 8) pubs in the dataset


In [5]:
#Postcode for testing
postcode = "SW1A 1AA"


In [6]:
import pgeocode
import requests

#Getting postcodes using postcode.io api
def get_coords(postcode):
    try:
        # Clean the postcode 
        clean_postcode = postcode.strip().replace(" ", "")
        
        # Call the free Postcodes.io API
        url = f"https://api.postcodes.io/postcodes/{clean_postcode}"
        response = requests.get(url)
        
        if response.status_code == 200:
            # Successfully found the exact postcode
            data = response.json()
            lat = data['result']['latitude']
            lon = data['result']['longitude']
            start_coords = [float(lat), float(lon)]
            print(start_coords)
            return start_coords
        else:
            # Postcode doesn't exist or API failed
            raise ValueError(f"Postcode '{clean_postcode}' not recognized by Postcodes.io")
            
    except Exception as e:
        print(f"Using default location. Geocoding error: {e}")
        start_coords = [51.5262, -0.1607]  # Fallback to LBS
        return start_coords


In [7]:
#Setting start coordinates in coordinate dictionary
start=get_coords(postcode)

coordinates['start']=start


[51.50101, -0.141563]


In [8]:
from sklearn.metrics.pairwise import haversine_distances
from math import radians
from numpy import sin, cos, arcsin, sqrt

def haversine(l1,l2):
    #Convert to radians
    lat1, lon1= map(radians,l1)
    lat2, lon2= map(radians, l2)

    dlat= lat1-lat2
    dlon= lon1-lon2 

    #Get radius of the earth (m)
    R = 6371

    #Calculate haversine
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * arcsin(sqrt(a))

    #Get harversine in meters
    diff= R*c
    return diff


n=len(coordinates)
ids= list(coordinates.keys())

matrix_coord = np.zeros((n,n))

for i in range(n):
    for j in range(n):
        if i!=j:
            l1=coordinates[ids[i]]
            l2=coordinates[ids[j]]
            matrix_coord[i][j]=round(haversine(l1,l2),2)

dist= pd.DataFrame(matrix_coord,index=ids,columns=ids)

start_pubs= dist_sorted=dist['start'].sort_values().head(101)
keep= start_pubs.index
dist_filtered= dist.loc[keep,keep]
d = dist_filtered.stack().to_dict()


In [9]:
dist.isna().sum().sum()


np.int64(0)

In [10]:
loc.head()


,id,lat,lon,addr:postcode,food,name,cuisine,opening_hours
0,451152,51.600840,-0.194608,N3 1DH,yes,King of Prussia,pizza;burger,Mo-Th 12:00-23:00; Fr-Sa 12:00-24:00; Su 12:00...
1,451154,51.599579,-0.196028,NaN,yes,The Catcher in the Rye,NaN,NaN
2,451271,51.614104,-0.176556,N12 0BP,NaN,The Tally Ho,NaN,NaN
3,12242503,51.592016,0.027962,NaN,NaN,Railway Bell,NaN,NaN
4,15262028,51.516481,-0.169827,W2 1JQ,yes,The Monkey Puzzle,NaN,Mo-Sa 12:00-23:30; Su 12:00-22:30


In [11]:
import folium
from folium.plugins import HeatMap 

base_map = folium.Map(location=[51.51,-0.12],control_scale=True, zoom_start=12)


# Rounding latitudes and longitudes to nearest 0.001 decimal
loc.loc[:,['lat', 'lon']] = loc.loc[:,['lat', 'lon']].round(3)
# Aggregating by latitude and longitude 
loc2= loc[['lat', 'lon']].groupby(['lat', 'lon']).count()
# Converting demand into a list
loc2 = loc2.reset_index().values.tolist()

# Add the desired heatmap to the base map
HeatMap(data= loc2,radius=0, max_zoom=15,minOpacity=0).add_to(base_map)
base_map


## Model Implementation

### Model Variables
The variables used are as follows:

| Variable | Domain | Description |
| :--- | :--- | :--- |
| $x_{ij}$ | Binary | 1 if the route travels from pub i to pub j else 0 |
| $y_i$ | Binary | 1 if pub i is visited else 0 |
| $p_i$ | Integer | Pints consumed at pub i (fixed at 1 per visited pub) |
| $u_i$ | Integer [1, n] | Visit-order variable for MTZ subtour elimination |
| W | Continuous | Total distance walked (km) |
| P | Integer | Total pints consumed |
| C | Continuous | Total spend (£) |
| T | Continuous | Total crawl time (minutes) |
| $\delta_{b1}, \delta_{b2}$ | Binary | Budget-threshold indicators |
| $\delta_t$ | Binary | Time-overrun indicator (T > $T_{max}$) |

### Objective Function
The objective function maximises a happiness score comprised of various components:
$$ \textbf{max.} \text{ Happiness} = \gamma\sqrt{P} - F(W,P) + \Phi(C) + \Psi(T) $$

| Component | Component Equation | Description |
| :--- | :--- | :--- |
| Pint Enjoyment | $\gamma\sqrt{P}$ | Linear pint enjoyment that rewards every visit to a pub with happiness scaled by $\gamma$ but has diminishing returns |
| Fatigue score | $F = \alpha W + \beta P^2$<br>where $\alpha$ is walking fatigue and $\beta$ is alcohol fatigue | The fatigue score decreases happiness as distance walked and pints consumed increases. Increasingly more points from happiness are deducted as more pints are consumed. |
| Budget | $\Phi(C) = -0.5 \delta_{b1}(C - 0.7B) - \delta_{b2}(C - 0.9B)$ | A piecewise function that captures negative impacts on happiness as spending increases. The penalty on happiness is initially moderate but becomes more severe when there is significant overspending. |
| Time Pressure Penalty | $\Psi(T) = -\lambda \delta_t(T - T_{max})$<br>Where $\lambda$ is time pressure | Happiness is deducted for every minute total crawl time exceeds budgeted time. |

In [12]:
m = gp.Model("Optimal_Pub_Crawl")
m.Params.NonConvex = 2 # Required for the P^2 and sqrt(P) operations
N_candidates= keep.tolist()[1:]
s = 'start'
V= N_candidates+[s]

T_max = 300      # Maximum available crawl time in minutes
v = 0.08         # Average walking speed (km/min)
c = 7            # Cost per pint (£7)
t_pub = 20      # Fixed dwell time per pub in minutes
budget_max = 80  # Maximum spend in £ (used by app.py slider; set default here for notebook)
M = 10000        # Big-M for linearisation
epsilon = 1e-4   # Small constant for strict inequality detection

# Objective Scaling Weights
alpha = 0.5      # Distance fatigue penalty
beta = 0.75       # Quadratic alcohol fatigue penalty 
gamma = 30.0     # Pint enjoyment scaling 
lam = 5.0        # Time pressure penalty multiplier

#Decision variables

# Route variables: 1 if travelling directly from location i to location j
x = m.addVars(V, V, vtype=GRB.BINARY, name="x")

# Visit variables: 1 if location i is visited
y = m.addVars(V, vtype=GRB.BINARY, name="y")

# Pints variable: Number of pints consumed at location i
p = m.addVars(V, vtype=GRB.INTEGER, name="p")

# Subtour elimination continuous variable (only applies to pubs)
u = m.addVars(N_candidates, vtype=GRB.INTEGER, lb=1, ub=n, name="u")

# Derived macro quantities
W = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="Total_Walked")
P = m.addVar(vtype=GRB.INTEGER, lb=0, name="Total_Pints")
C_spend = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="Total_Spend")
T_time = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="Total_Time")

# Non-linear function placeholders (Fatigue & Enjoyment)
P_sq = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="P_Squared")
P_sqrt = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="P_Sqrt")

# Binary identifiers for penalty linearisation
delta1 = m.addVar(vtype=GRB.BINARY, name="delta1_budget70")
delta2 = m.addVar(vtype=GRB.BINARY, name="delta2_budget90")
delta3 = m.addVar(vtype=GRB.BINARY, name="delta3_time")

#Derived quantities

# 4.1 Total Distance Walked
m.addConstr(W == gp.quicksum(d[i, j] * x[i, j] for i in V for j in V if i != j), name="Calc_W")

# Total Pints consumed across the crawl
# (We force p[s] to 0 below, so LBS doesn't contribute to pints)
m.addConstr(P == gp.quicksum(p[i] for i in N_candidates), name="Calc_P")

# 4.2 Total Spend
m.addConstr(C_spend == c * P, name="Calc_C")

# 4.3 Total Time
# Note: We only add t_pub for candidate pubs visited, not the starting location LBS.
m.addConstr(T_time == (W / v) + t_pub * gp.quicksum(y[i] for i in N_candidates), name="Calc_T")

# Non-linear equality mappings for P² and √P
m.addConstr(P_sq == P * P, name="Calc_P_sq")
m.addGenConstrPow(P, P_sqrt, 0.5, name="Calc_P_sqrt")

Set parameter Username
Set parameter LicenseID to value 2761735
Academic license - for non-commercial use only - expires 2027-01-07
Set parameter NonConvex to value 2


<gurobi.GenConstr *Awaiting Model Update*>

### Constraints

**No repeats (C1)**: Each pub in the optimal route must be visited exactly once.

**Starting point (C2)**:  The optimal route always starts and ends at the starting location, but no pints are consumed there. 

**Subtour elimination (C3)**: Miller–Tucker–Zemlin (MTZ) inequalities prevent disconnected sub-loops by enforcing a consistent visit ordering.

**Pint assignment (C4)**: Each visited pub contributes exactly one pint (pᵢ = yᵢ). This simplification keeps the model tractable; the formulation generalises to a bounded range.

**Budget and time penalty linearisation (C5–C6)**: Big-M constraints activate the binary indicators δb1, δb2, δt when spending or time exceeds their respective thresholds. The user-inputted budget is not a hard limit on how much the user can spend. Rather, once the money spent on the pub crawl exceeds the budget, the integer program punishes additional spending by decreasing happiness. There are two thresholds for spending: a soft threshold at 70% of Bmax  (δb1) and a hard threshold at 90% of Bmax (δb2). This captures the additional stress experienced when running out of budget. 
Similarly, the constraint punishes happiness when the time spent on the pub crawl exceeds the user inputted time. For every minute spent above the maximum budgeted time, happiness would be deducted by λ.


In [13]:


#Constraints

# (C1) Each candidate pub visited at most once
for i in N_candidates:
    m.addConstr(gp.quicksum(x[i, j] for j in V if i != j) == y[i], name=f"FlowOut_{i}")
    m.addConstr(gp.quicksum(x[j, i] for j in V if i != j) == y[i], name=f"FlowIn_{i}")

# (C2) Route starts and ends from fixed origin s ('LBS')
m.addConstr(gp.quicksum(x[s, j] for j in N_candidates) == 1, name="DepartOrigin")
m.addConstr(gp.quicksum(x[i, s] for i in N_candidates) == 1, name="ReturnOrigin")
m.addConstr(y[s] == 1, name="VisitOrigin")
m.addConstr(p[s] == 0, name="NoPintsAtOrigin") # No pints allowed at the starting point

# (C3) Subtour Elimination (Miller–Tucker–Zemlin)
for i in N_candidates:
    for j in N_candidates:
        if i != j:
            m.addConstr(u[i] - u[j] + n * x[i, j] <= n - 1, name=f"MTZ_{i}_{j}")

# (C4) Pints per pub bounds
for i in N_candidates:
    m.addConstr(p[i] == y[i], name=f"ExactPints_{i}")

# (C5) Budget penalty linearisation
soft_threshold = 0.7 * budget_max
hard_threshold = 0.9 * budget_max
m.addConstr(C_spend - soft_threshold <= M * delta1, name="Budget1_UB")
m.addConstr(C_spend - soft_threshold >= epsilon * delta1 - M * (1 - delta1), name="Budget1_LB")

m.addConstr(C_spend - hard_threshold <= M * delta2, name="Budget2_UB")
m.addConstr(C_spend - hard_threshold >= epsilon * delta2 - M * (1 - delta2), name="Budget2_LB")

# (C6) Time penalty linearisation
m.addConstr(T_time - T_max <= M * delta3, name="Time_UB")
m.addConstr(T_time - T_max >= epsilon * delta3 - M * (1 - delta3), name="Time_LB")

# (C7) Min. pubs visited
#min_pubs_wanted = 4
#m.addConstr(P >= min_pubs_wanted, name="Force_Min_Pubs")

# ==========================================
# 7. Objective Function
# ==========================================

# 5.1 Fatigue Score
F = alpha * W + beta * P_sq

# 5.3 Budget Penalty
Phi = -0.5 * (C_spend - soft_threshold) * delta1 - 1.0 * (C_spend - 90) * delta2

# 5.4 Time Penalty
Psi = -lam * (T_time - T_max) * delta3

# 5.5 Full Happiness Objective
H = m.addVar(vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY,name="Happiness")

# Add a constraint linking the variable H to the happiness formula
#m.addConstr(H == gamma * P_sqrt - F + Phi + Psi, name="Calc_Happiness")

#m.setObjective(H, GRB.MAXIMIZE)
m.setObjective(gamma * P - F + Phi + Psi, GRB.MAXIMIZE)

# ==========================================
# 8. Execution and Output
# ==========================================
m.update()
m.Params.TimeLimit = 60
m.optimize()

if m.status == GRB.OPTIMAL or (m.status == GRB.TIME_LIMIT and m.SolCount > 0):
    
    if m.status == GRB.TIME_LIMIT:
        print("\n  TIME LIMIT REACHED: Displaying the best solution found so far.")
    else:
        print("\n OPTIMAL PUB CRAWL FOUND")

    # --- CALCULATE COMPONENT SCORES ---
    total_joy = gamma * P_sqrt.X
    walk_penalty = alpha * W.X
    alcohol_penalty = beta * P_sq.X

    print(f"Total Happiness: {m.objVal:.2f}")
    print(f"  └─ Joy: +{total_joy:.2f} | Walk: -{walk_penalty:.2f} | Alcohol: -{alcohol_penalty:.2f}")
    print(f"Total Distance:  {W.X:.2f} km")
    print(f"Total Time:      {T_time.X:.2f} mins")
    print(f"Total Spend:     £{C_spend.X}")
    print(f"Total Pints:     {int(P.X)} pints")
    
    print("\nRoute & Pints:")
    current = s
    visited_count = 0
    
    # Use a safety break in the loop to prevent infinite loops if x is messy
    max_visits = len(V) 
    
    while visited_count < max_visits:
        # Find next location
        try:
            next_loc = [j for j in V if current != j and x[current, j].X > 0.5][0]
        except IndexError:
            print(" -> [Error: Route broken - no further connection found]")
            break
        
        if next_loc == s:
            print(" -> Return to LBS")
            break
            
        pub_name = pubs[next_loc].get('tags', {}).get('name', f'Pub {next_loc}')
        print(f" -> Visit {pub_name} (ID: {next_loc}) | Pints consumed: {int(p[next_loc].X)}")
        
        current = next_loc
        visited_count += 1

elif m.status == GRB.TIME_LIMIT and m.SolCount == 0:
    print("\n❌ TIME LIMIT REACHED: No feasible solution was found in the allotted time.")
    print("Try increasing the TimeLimit or simplifying the constraints.")

elif m.status == GRB.INFEASIBLE:
    print("\n❌ MODEL IS INFEASIBLE: No solution exists that satisfies all constraints.")
    # (Optional: call m.computeIIS() here if on a powerful machine)

else:
    print(f"\nOptimization stopped with status code: {m.status}")


Set parameter TimeLimit to value 60
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F80)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  60
NonConvex  2

Optimize a model with 10214 rows, 10513 columns and 60620 nonzeros (Max)
Model fingerprint: 0x9079c8f4
Model has 6 linear objective coefficients
Model has 3 quadratic objective terms
Model has 1 quadratic constraint
Model has 1 function constraint treated as nonlinear
  1 POW
Variable types: 6 continuous, 10507 integer (10305 binary)
Coefficient statistics:
  Matrix range     [2e-02, 1e+04]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [5e-01, 2e+03]
  QObjective range [1e+00, 1e+01]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+04]
  NLCon coe range  [1e+00, 1e+00]

Presolve removed 102 rows and 205 columns
Presolve time: 0.08s
Presolved: 10120 rows, 10312 colum

In [ ]:
# 1. Initialize the map centered on Buckingham Palace
# Convert the coordinates to a list to prevent pandas indexing errors
start_coords = list(coordinates[s]) 
m_map = folium.Map(location = start_coords, zoom_start=15, tiles="cartodbpositron")

# 2. Add the starting point marker
folium.Marker(
    location=start_coords,
    popup="START: ",
    icon=folium.Icon(color="red", icon="home")
).add_child(folium.Tooltip("Origin (LBS)")).add_to(m_map)

# 3. Trace the route and add pub markers
current = s
path_coords = [start_coords]
visited_order = 0

while True:
    # Find the next location in the optimal route where x[current, j] == 1
    next_loc = [j for j in V if current != j and x[current, j].X > 0.5][0]
    
    # Get coordinates for the next location AND cast to list
    next_coords = list(coordinates[next_loc])
    path_coords.append(next_coords)
    
    if next_loc == s:
        break # Back at start
    
    visited_order += 1
    
    # Get Pub metadata for the popup
    pub_info = pubs.get(next_loc, {})
    pub_name = pub_info.get('tags', {}).get('name', f"Pub {next_loc}")
    pint_count = int(p[next_loc].X)
    
    # Add Marker for the Pub
    folium.Marker(
        location=next_coords,
        popup=f"<b>Stop {visited_order}: {pub_name}</b><br>Pints: {pint_count}",
        icon=folium.Icon(color="blue", icon="glass", prefix="fa")
    ).add_child(folium.Tooltip(f"{visited_order}. {pub_name}")).add_to(m_map)
    
    current = next_loc

# 4. Draw the Path (PolyLine)
folium.PolyLine(
    locations=path_coords,
    color="orange",
    weight=5,
    opacity=0.7,
    dash_array='10' # Dashed line to indicate walking path
).add_to(m_map)

# 5. Display the map
m_map

In [15]:
# 1. Setup the Map
m_map = folium.Map(location=list(coordinates[s]), zoom_start=14, tiles="cartodbpositron")

# 2. Tracking Variables
current = s
cumulative_pints = 0
cumulative_dist = 0
visited_order = 0

# 3. Add Start Marker
folium.Marker(list(coordinates[s]), popup="START: ", icon=folium.Icon(color="gray", icon="home")).add_to(m_map)

while True:
    try:
        # Find next location in the optimal route
        next_loc = [j for j in V if current != j and x[current, j].X > 0.5][0]
    except IndexError: break
    
    # Calculate stats for this LEG
    leg_dist = d[current, next_loc]
    cumulative_dist += leg_dist
    
    # Calculate "Mood" at this point in time
    # (Using your model parameters: alpha, beta, gamma)
    joy = gamma * np.sqrt(cumulative_pints)
    fatigue = (beta * (cumulative_pints**2)) + (alpha * cumulative_dist)
    net_happiness = joy - fatigue
    line_weight = 3 + (cumulative_pints * 3)

    # --- COLOR LOGIC FOR THE LINE SEGMENT ---
    # We color the line based on how they feel LEAVING the current pub
    if net_happiness < 0:
        path_color = "#e74c3c" # Red (Struggling)
    elif fatigue > (joy * 0.5):
        path_color = "#e67e22" # Orange (Tiring)
    else:
        path_color = "#27ae60" # Green (Energetic)

    # 4. DRAW THE INTOXICATED PATH SEGMENT
    # Notice both coordinates are wrapped in list()
    folium.PolyLine(
        locations=[list(coordinates[current]), list(coordinates[next_loc])],
        color=path_color,
        weight=line_weight, 
        opacity=0.6,
        tooltip=f"Intoxication Level: {cumulative_pints} pints"
    ).add_to(m_map)

    # If next_loc is the start, we've completed the loop
    if next_loc == s: break 

    # 5. Add Pub Marker
    visited_order += 1
    pints_here = int(p[next_loc].X)
    cumulative_pints += pints_here
    
    pub_info = pubs.get(next_loc, {})
    pub_name = pub_info.get('tags', {}).get('name', f"Pub {next_loc}")

    folium.Marker(
        location=list(coordinates[next_loc]),
        popup=f"<b>Stop {visited_order}: {pub_name}</b><br>Pints: {pints_here}<br>Total Score: {net_happiness:.2f}",
        icon=folium.Icon(color="blue", icon="glass", prefix="fa")
    ).add_to(m_map)
    
    current = next_loc

m_map

---
## Final Streamlit Prototype


In [16]:
%%writefile app.py
import json
import numpy as np
import pandas as pd
import folium
from folium.plugins import PolyLineTextPath
import pgeocode
import gurobipy as gp
from gurobipy import GRB
from math import radians
from numpy import sin, cos, arcsin, sqrt
import streamlit as st
import streamlit.components.v1 as components
from OSMPythonTools.nominatim import Nominatim
import requests
from OSMPythonTools.overpass import Overpass, overpassQueryBuilder
import branca.colormap as cm


# ════════════════════════════════════════
# DATA LOADING
# ════════════════════════════════════════
def load_pub_data(postcode):
    #with open("interpreter.json") as f:
        #raw = json.load(f)
    overpass = Overpass()
    nominatim=Nominatim()

    query = overpassQueryBuilder(
        area=nominatim.query('London'), 
        elementType=['node', 'way', 'relation'], 
        selector='"amenity"="pub"', 
        includeGeometry=True,
        out='body'
        )

    result=overpass.query(query)
    data=result.toJSON()
    pubs = {elem["id"]: elem for elem in data["elements"] if elem["type"] == "node"}
    df = pd.json_normalize(data["elements"])
    loc = df.copy()
    loc.columns = [col.replace("tags.", "") for col in loc]
    loc = loc.loc[(loc["type"] == "node").values, :]
    loc = loc[["id", "lat", "lon", "addr:postcode", "food", "name", "cuisine", "opening_hours"]]
    loc_cleaned = loc[loc["name"].notna()]
    pub_id = loc_cleaned["id"].astype(int).tolist()
    coordinates = (
        loc_cleaned[loc_cleaned["id"].isin(pub_id)]
        .set_index("id")[["lat", "lon"]]
        .T.to_dict("list")
    )
    try:
        # Clean the postcode (Postcodes.io prefers no spaces)
        clean_postcode = postcode.strip().replace(" ", "")
        
        # Call the free Postcodes.io API
        url = f"https://api.postcodes.io/postcodes/{clean_postcode}"
        response = requests.get(url)
        
        if response.status_code == 200:
            # Successfully found the exact postcode
            data = response.json()
            lat = data['result']['latitude']
            lon = data['result']['longitude']
            start_coords = [float(lat), float(lon)]
        else:
            # Postcode doesn't exist or API failed
            raise ValueError(f"Postcode '{clean_postcode}' not recognized by Postcodes.io")
            
    except Exception as e:
        st.warning(f"Using default location. Geocoding error: {e}")
        start_coords = [51.5262, -0.1607]  # Fallback to LBS

    coordinates["start"] = start_coords

    def haversine(l1, l2):
        lat1, lon1 = map(radians, l1)
        lat2, lon2 = map(radians, l2)
        dlat, dlon = lat1 - lat2, lon1 - lon2
        a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
        return 6371 * 2 * arcsin(sqrt(a))

    ids = list(coordinates.keys())
    n_total = len(ids)
    matrix_coord = np.zeros((n_total, n_total))
    for i in range(n_total):
        for j in range(n_total):
            if i != j:
                matrix_coord[i][j] = round(haversine(coordinates[ids[i]], coordinates[ids[j]]), 2)
    dist = pd.DataFrame(matrix_coord, index=ids, columns=ids)
    keep = dist["start"].sort_values().head(101).index
    d = dist.loc[keep, keep].stack().to_dict()
    s = "start"
    N_candidates = keep.tolist()[1:]
    V = N_candidates + [s]
    n = len(N_candidates)
    return {"coordinates": coordinates, "pubs": pubs, "d": d,
            "N_candidates": N_candidates, "s": s, "V": V, "n": n}


# ════════════════════════════════════════
# MODEL
# ════════════════════════════════════════
def run_model(data, T_max, t_pub, v, budget_max, alpha, beta, gamma, lam):
    N_candidates = data["N_candidates"]
    s = data["s"]; V = data["V"]; d = data["d"]; n = data["n"]; pubs = data["pubs"]
    c = 7; M = 10000; epsilon = 1e-4

    m = gp.Model("Optimal_Pub_Crawl")
    m.Params.OutputFlag = 0
    m.Params.NonConvex = 2
    m.Params.TimeLimit = 60

    x      = m.addVars(V, V, vtype=GRB.BINARY,     name="x")
    y      = m.addVars(V,    vtype=GRB.BINARY,     name="y")
    p      = m.addVars(V,    vtype=GRB.INTEGER,    name="p")
    u      = m.addVars(N_candidates, vtype=GRB.INTEGER, lb=1, ub=n, name="u")
    W      = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="Total_Walked")
    P      = m.addVar(vtype=GRB.INTEGER,    lb=0,   name="Total_Pints")
    C_spend= m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="Total_Spend")
    T_time = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="Total_Time")
    P_sq   = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="P_Squared")
    P_sqrt = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="P_Sqrt")
    delta1 = m.addVar(vtype=GRB.BINARY, name="delta1")
    delta2 = m.addVar(vtype=GRB.BINARY, name="delta2")
    delta3 = m.addVar(vtype=GRB.BINARY, name="delta3")

    m.addConstr(W == gp.quicksum(d[i, j] * x[i, j] for i in V for j in V if i != j))
    m.addConstr(P == gp.quicksum(p[i] for i in N_candidates))
    m.addConstr(C_spend == c * P)
    m.addConstr(T_time == (W / v) + t_pub * gp.quicksum(y[i] for i in N_candidates))
    m.addConstr(P_sq == P * P)
    m.addGenConstrPow(P, P_sqrt, 0.5)
    for i in N_candidates:
        m.addConstr(gp.quicksum(x[i, j] for j in V if i != j) == y[i])
        m.addConstr(gp.quicksum(x[j, i] for j in V if i != j) == y[i])
    m.addConstr(gp.quicksum(x[s, j] for j in N_candidates) == 1)
    m.addConstr(gp.quicksum(x[i, s] for i in N_candidates) == 1)
    m.addConstr(y[s] == 1)
    m.addConstr(p[s] == 0)
    for i in N_candidates:
        m.addConstr(p[i] == y[i])
    for i in N_candidates:
        for j in N_candidates:
            if i != j:
                m.addConstr(u[i] - u[j] + n * x[i, j] <= n - 1)
    soft = 0.7 * budget_max
    hard = 0.9 * budget_max
    m.addConstr(C_spend - soft <= M * delta1)
    m.addConstr(C_spend - soft >= epsilon * delta1 - M * (1 - delta1))
    m.addConstr(C_spend - hard <= M * delta2)
    m.addConstr(C_spend - hard >= epsilon * delta2 - M * (1 - delta2))
    m.addConstr(T_time - T_max <= M * delta3)
    m.addConstr(T_time - T_max >= epsilon * delta3 - M * (1 - delta3))
    F   = alpha * W + beta * P_sq
    Phi = -0.5 * (C_spend - soft) * delta1 - 1.0 * (C_spend - hard) * delta2
    Psi = -lam * (T_time - T_max) * delta3
    m.setObjective(gamma * P_sqrt - F + Phi + Psi, GRB.MAXIMIZE)
    m.optimize()

    if m.SolCount == 0:
        return None

    route = []
    current = s
    for _ in range(len(V)):
        try:
            next_loc = [j for j in V if current != j and x[current, j].X > 0.5][0]
        except IndexError:
            break
        if next_loc == s:
            break

        # Extract tags
        pub_tags = pubs.get(next_loc, {}).get("tags", {})
        pub_name = pub_tags.get("name", f"Pub {next_loc}")
        
        # New: metadata extraction
        food_info = pub_tags.get("food", "Not specified")
        cuisine_info = pub_tags.get("cuisine", "Traditional")

        route.append({"stop": len(route) + 1, "id": next_loc, "name": pub_name, "food": food_info, "cuisine": cuisine_info, "pints": int(p[next_loc].X)})
        current = next_loc

    x_vals = {(i, j): x[i, j].X for i in V for j in V if i != j and x[i, j].X > 0.5}
    return {"obj": round(m.objVal, 2), "pubs_visited": len(route), "pints": int(P.X),
            "distance": round(W.X, 2), "spend": round(C_spend.X, 2),
            "time": round(T_time.X, 1), "route": route, "x_vals": x_vals}


# ════════════════════════════════════════
# MAP
# ════════════════════════════════════════

fatigue_map = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=100)

def build_map(result, data, alpha, beta, gamma):
    coordinates = data["coordinates"]; V = data["V"]
    d = data["d"]; s = data["s"]; x_vals = result["x_vals"]

    ratio_map = cm.LinearColormap(
        colors=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'], # Green, Yellow, Orange, Red
        index=[0, 0.25, 0.50, 0.75], 
        vmin=0, 
        vmax=1.0 # Values above 1.0 (Fatigue > Joy) will stay Red
    )
    ratio_map.caption = 'Fatigue as % of Joy (Green < 25%, Red > 75%)'

    m_map = folium.Map(
        location=coordinates[s],
        zoom_start=14,
        tiles="cartodbpositron",
        zoom_control=False,
        attributionControl=False,
    )

    # ── Start marker (custom DivIcon) ──
    start_icon = folium.DivIcon(
        icon_size=(32, 32),
        icon_anchor=(16, 16),
        html='''
        <div style="
            width:32px; height:32px; border-radius:50%;
            background:#0f0f0f; border:3px solid #fff;
            display:flex; align-items:center; justify-content:center;
            box-shadow: 0 2px 8px rgba(0,0,0,0.25);
            font-size:14px; color:#fff; line-height:1;
        ">⌂</div>
        '''
    )
    folium.Marker(
        coordinates[s],
        icon=start_icon,
        tooltip="Start",
    ).add_to(m_map)

    # ── Trace route, draw segments with arrows ──
    current = s
    cumulative_pints = 0
    cumulative_dist = 0
    visited_order = 0

    fatigue_map.caption = 'Walking Fatigue Level'
    m_map.add_child(fatigue_map)

    while True:
        next_list = [j for j in V if current != j and x_vals.get((current, j), 0) > 0.5]
        if not next_list:
            break
        next_loc = next_list[0]
        leg_dist = d.get((current, next_loc), 0)
        cumulative_dist += leg_dist

        joy = gamma * np.sqrt(max(cumulative_pints, 0))
        fatigue = (beta * (cumulative_pints ** 2)) + (alpha * cumulative_dist)
        fatigue_ratio = fatigue / joy if joy > 0 else 0
        net_happiness = joy - fatigue

        #line_weight = max(3, 3 + cumulative_pints * 3)
        #path_color = (
            #"#e74c3c" if net_happiness < 0
            #else "#f39c12" if fatigue > joy * 0.5
            #else "#2ecc71"
        #)

        path_color = ratio_map(min(fatigue_ratio, 1.0))

        # Draw the line segment
        line = folium.PolyLine(
            locations=[coordinates[current], coordinates[next_loc]],
            color=path_color, 
            #weight=line_weight, 
            #opacity=line_opacity
        ).add_to(m_map)

        # Add directional arrows along the segment
        arrow = PolyLineTextPath(
            line,
            text="\u25B6",          # ► arrow character
            repeat=True,
            offset=12,
            attributes={
                "fill": path_color,
                "font-size": "11",
                "font-weight": "bold",
                "dy": "5",
            },
        )
        arrow.add_to(m_map)

        if next_loc == s:
            break

        # ── Pub marker (numbered circle) ──
        visited_order += 1
        stop_info = next((r for r in result["route"] if r["id"] == next_loc), {})
        pints_here = stop_info.get("pints", 0)
        pub_name = stop_info.get("name", str(next_loc))
        cumulative_pints += pints_here

        pub_icon = folium.DivIcon(
            icon_size=(28, 28),
            icon_anchor=(14, 14),
            html=f'''
            <div style="
                width:28px; height:28px; border-radius:50%;
                background:#fff; border:2.5px solid {path_color};
                display:flex; align-items:center; justify-content:center;
                box-shadow: 0 1px 6px rgba(0,0,0,0.18);
                font-size:12px; font-weight:700; color:{path_color};
                font-family: -apple-system, BlinkMacSystemFont, sans-serif;
            ">{visited_order}</div>
            '''
        )

        popup_html = f"""
        <div style="font-family:-apple-system,BlinkMacSystemFont,sans-serif;font-size:13px;line-height:1.5;min-width:140px;">
            <div style="font-weight:700;font-size:14px;margin-bottom:4px;">{pub_name}</div>
            <div style="color:#888;font-size:11px;text-transform:uppercase;letter-spacing:0.05em;">Stop {visited_order}</div>
        </div>
        """

        folium.Marker(
            location=coordinates[next_loc],
            icon=pub_icon,
            popup=folium.Popup(popup_html, max_width=200),
            tooltip=f"{visited_order}. {pub_name}",
        ).add_to(m_map)

        current = next_loc

    return m_map


# ════════════════════════════════════════
# STREAMLIT UI
# ════════════════════════════════════════
st.set_page_config(
    page_title="Pub Crawl Optimiser",
    layout="wide",
    initial_sidebar_state="collapsed",
)

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:ital,opsz,wght@0,9..40,300;0,9..40,400;0,9..40,500;0,9..40,600;0,9..40,700&display=swap');

/* ── Reset ── */
* { font-family: 'DM Sans', -apple-system, BlinkMacSystemFont, sans-serif !important; }
#MainMenu, footer, header, [data-testid="collapsedControl"] { display: none !important; }

/* ── Canvas ── */
.stApp { background: #fafaf9; color: #1a1a1a; }
.block-container {
    max-width: 760px; margin: 0 auto;
    padding: 3.5rem 2rem 5rem 2rem;
}

/* ── Title ── */
.app-title {
    font-size: 1.5rem !important; font-weight: 700 !important;
    letter-spacing: -0.035em !important; color: #1a1a1a !important;
    margin-bottom: 0.15rem !important;
}
.app-sub {
    font-size: 0.82rem; color: #999; font-weight: 400;
    margin-bottom: 2rem; line-height: 1.6;
}

/* ── Text input ── */
.stTextInput > label { display: none; }
.stTextInput input {
    background: #fff !important; border: 1.5px solid #e5e5e3 !important;
    border-radius: 10px !important; font-size: 0.9rem !important;
    padding: 0.7rem 1rem !important; color: #1a1a1a !important;
    box-shadow: none !important; transition: border-color 0.2s;
}
.stTextInput input:focus {
    border-color: #1a1a1a !important; background: #fff !important; box-shadow: none !important;
}
.stTextInput input::placeholder { color: #c0c0c0 !important; }

/* ── Control labels ── */
.ctrl-label {
    font-size: 0.6rem; font-weight: 700; letter-spacing: 0.12em;
    text-transform: uppercase; color: #b0b0b0;
    margin-bottom: 0.5rem; margin-top: 0.25rem; display: block;
}

/* ── Sliders ── */
.stSlider > label { font-size: 0.78rem !important; font-weight: 500 !important; color: #555 !important; }
[data-baseweb="slider"] [data-testid="stThumb"] { background: #1a1a1a !important; border: none !important; }
[data-baseweb="slider"] [data-testid="stTrackFill"] { background: #1a1a1a !important; }

/* ── Radio buttons as pill selectors ── */
.stRadio > label { display: none !important; }

div[data-testid="stRadio"] > div[role="radiogroup"] {
    display: flex !important; flex-direction: column !important; gap: 0.3rem !important;
}
div[data-testid="stRadio"] label[data-baseweb="radio"] {
    display: flex !important; align-items: center !important;
    padding: 0.5rem 0.85rem !important; border-radius: 8px !important;
    border: 1.5px solid #e5e5e3 !important; background: #fff !important;
    color: #666 !important; font-size: 0.78rem !important; font-weight: 500 !important;
    cursor: pointer !important; transition: all 0.15s !important; width: 100% !important;
}
div[data-testid="stRadio"] label[data-baseweb="radio"] div[data-testid="stMarkdownContainer"] p {
    color: #777 !important; margin-bottom: 0 !important;
}
div[data-testid="stRadio"] > div > label:hover {
    border-color: #1a1a1a !important;
}
div[data-testid="stRadio"] > div > label:has(input:checked) {
    background: #1a1a1a !important; border-color: #1a1a1a !important;
}
div[data-testid="stRadio"] label[data-baseweb="radio"]:has(input:checked) div[data-testid="stMarkdownContainer"] p {
    color: #fff !important;
}
div[data-testid="stRadio"] > div > label > div:first-child { display: none !important; }

/* ── Button ── */
div[data-testid="stButton"] > button {
    background: #1a1a1a !important; color: #fff !important; border: none !important;
    border-radius: 10px !important; font-size: 0.85rem !important; font-weight: 600 !important;
    padding: 0.7rem 1.5rem !important; width: 100% !important;
    margin-top: 1rem !important; transition: opacity 0.15s !important;
    box-shadow: none !important; letter-spacing: -0.01em;
}
div[data-testid="stButton"] > button:hover { opacity: 0.75 !important; }

/* ── Divider ── */
.divider { border: none; border-top: 1px solid #ececea; margin: 1.75rem 0; }

/* ── Metric cards ── */
[data-testid="metric-container"] {
    background: #fff; border: 1.5px solid #ececea;
    border-radius: 12px; padding: 1rem 0.75rem 0.85rem;
}
[data-testid="stMetricLabel"] {
    font-size: 0.58rem !important; font-weight: 700 !important;
    letter-spacing: 0.12em !important; text-transform: uppercase !important; color: #b0b0b0 !important;
}
[data-testid="stMetricValue"] {
    font-size: 1.35rem !important; font-weight: 700 !important;
    color: #1a1a1a !important; letter-spacing: -0.03em !important;
}

/* ── Data table ── */
[data-testid="stDataFrame"] {
    border: 1.5px solid #ececea !important; border-radius: 12px !important; overflow: hidden !important;
}
[data-testid="stDataFrame"] thead tr th {
    background: #f5f5f4 !important; color: #b0b0b0 !important;
    font-size: 0.6rem !important; font-weight: 700 !important;
    letter-spacing: 0.1em !important; text-transform: uppercase !important;
    border-bottom: 1px solid #ececea !important; padding: 0.55rem 1rem !important;
}
[data-testid="stDataFrame"] tbody tr td {
    color: #333 !important; border-bottom: 1px solid #f5f5f4 !important; padding: 0.55rem 1rem !important;
    font-size: 0.82rem !important;
}
[data-testid="stDataFrame"] tbody tr:last-child td { border-bottom: none !important; }

/* ── Section labels ── */
.section-label {
    font-size: 0.6rem; font-weight: 700; letter-spacing: 0.12em;
    text-transform: uppercase; color: #b0b0b0; margin-bottom: 0.65rem;
}

/* ── Map iframe ── */
iframe { border-radius: 12px; border: 1.5px solid #ececea; }

/* ── Spinner ── */
.stSpinner p { font-size: 0.82rem; font-weight: 500; color: #999; }

/* ── Notifications ── */
[data-testid="stNotificationContent"] { color: #1a1a1a !important; font-weight: 500; }

/* ── Legend ── */
.map-legend {
    display: flex; gap: 1.25rem; margin-top: 0.65rem; margin-bottom: 0.25rem;
}
.map-legend-item {
    display: flex; align-items: center; gap: 0.35rem;
    font-size: 0.7rem; color: #999; font-weight: 500;
}
.map-legend-dot {
    width: 8px; height: 8px; border-radius: 50%;
}
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)

# ── Header ────────────────────────────────────────────────────────────────
st.markdown('<p class="app-title">Pub Crawl Optimiser</p>', unsafe_allow_html=True)
st.markdown(
    '<p class="app-sub">Enter your postcode and set your preferences. '
    'We\'ll find the optimal route through nearby pubs, balancing distance, '
    'time, and how hard you want to go.</p>',
    unsafe_allow_html=True
)

# ── Postcode ──────────────────────────────────────────────────────────────
def format_postcode(pc):
    pc = pc.replace(" ", "").upper()
    if len(pc) > 3:
        return f"{pc[:-3]} {pc[-3:]}"
    return pc

postcode = str(st.text_input("postcode", placeholder="Postcode — e.g. W1B 1AH", label_visibility="collapsed", key="pc_input"))

if postcode:
    formatted = format_postcode(postcode)

# ── Settings ──────────────────────────────────────────────────────────────
col1, col2, col3 = st.columns(3)

with col1:
    st.markdown('<span class="ctrl-label">Time & pace</span>', unsafe_allow_html=True)
    T_max = st.slider("How long do you have?", 60, 480, 240, step=30, format="%d min")
    pace = st.radio("Walking pace", ["Leisurely", "Normal", "Brisk"], index=1)
    v = {"Leisurely": 0.05, "Normal": 0.08, "Brisk": 0.12}[pace]

with col2:
    st.markdown('<span class="ctrl-label">Pub style</span>', unsafe_allow_html=True)
    t_pub = st.slider("How long per pub?", 15, 60, 20, step=5, format="%d min")
    walk_pref = st.radio("Walking tolerance", ["Prefer close pubs", "Happy to walk", "Love exploring"], index=1)
    alpha = {"Prefer close pubs": 1.5, "Happy to walk": 0.5, "Love exploring": 0.1}[walk_pref]

with col3:
    st.markdown('<span class="ctrl-label">Budget</span>', unsafe_allow_html=True)
    budget_max = st.slider("Max spend tonight", 20, 200, 80, step=10, format="£%d")
    stamina = st.radio("Drinking stamina", ["Light night", "Standard", "Going all in"], index=1)
    beta = {"Light night": 1.5, "Standard": 0.75, "Going all in": 0.2}[stamina]

GAMMA = 30.0
LAM   = 5.0

# ── Run ───────────────────────────────────────────────────────────────────
run = st.button("Plan my crawl")

if run:
    if not postcode.strip():
        st.warning("Please enter a postcode.")
        st.stop()
    with st.spinner("Finding pubs near you..."):
        try:
            data = load_pub_data(postcode.strip())
        except FileNotFoundError:
            st.error("interpreter.json not found — place it in the same folder as app.py.")
            st.stop()
        except Exception as e:
            st.error(f"Error: {e}")
            st.stop()
    with st.spinner("Optimising your route..."):
        result = run_model(data, T_max=T_max, t_pub=t_pub, v=v, budget_max=budget_max,
                           alpha=alpha, beta=beta, gamma=GAMMA, lam=LAM)
    if result is None:
        st.error("No route found — try increasing your time or budget.")
        st.stop()
    st.session_state["result"] = result
    st.session_state["data"]   = data
    st.session_state["params"] = {"alpha": alpha, "beta": beta, "gamma": GAMMA}

# ── Results ───────────────────────────────────────────────────────────────
if "result" in st.session_state:
    result = st.session_state["result"]
    data   = st.session_state["data"]
    params = st.session_state["params"]

    st.markdown('<hr class="divider">', unsafe_allow_html=True)
    st.markdown('<p class="app-title">Your route</p>', unsafe_allow_html=True)

    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Pubs",     result["pubs_visited"])
    c2.metric("Pints",    result["pints"])
    c3.metric("Distance", f"{result['distance']:.1f} km")
    c4.metric("Time",     f"{result['time']:.0f} min")
    c5.metric("Spend",    f"£{result['spend']:.0f}")

    st.markdown('<hr class="divider">', unsafe_allow_html=True)

    m_map = build_map(result, data, **params)
    map_html = m_map._repr_html_()
    components.html(map_html, height=480)

    # Map legend
    st.markdown('''
    <div class="map-legend">
        <div class="map-legend-item"><div class="map-legend-dot" style="background:#2ecc71;"></div> Energetic</div>
        <div class="map-legend-item"><div class="map-legend-dot" style="background:#f39c12;"></div> Tiring</div>
        <div class="map-legend-item"><div class="map-legend-dot" style="background:#e74c3c;"></div> Struggling</div>
    </div>
    ''', unsafe_allow_html=True)

    st.markdown('<hr class="divider">', unsafe_allow_html=True)

    st.markdown('<p class="section-label">Stop by stop</p>', unsafe_allow_html=True)
    df_route = pd.DataFrame(result["route"])[["stop", "name", "cuisine"]]
    df_route.columns = ["Stop", "Pub", "Cuisine"]
    st.dataframe(df_route, use_container_width=True, hide_index=True)


Overwriting app.py


In [17]:
%%writefile requirements.txt
streamlit>=1.32.0
streamlit-folium>=0.20.0
gurobipy>=11.0.0
pgeocode>=0.4.0
folium>=0.16.0
pandas>=2.0.0, <3.0.0
numpy>=1.26.0
scikit-learn>=1.4.0


Overwriting requirements.txt


In [18]:
import subprocess, webbrowser, time, sys

# Use sys.executable to force Streamlit to run in this exact virtual environment
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.headless', 'true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
webbrowser.open('http://localhost:8501')
print('App running at http://localhost:8501')
print('To stop: proc.terminate()')

App running at http://localhost:8501
To stop: proc.terminate()
